In [2]:
# Import numpy for handling tables and arrays
import numpy as np
# Import gymnasium for the Mountain Car physics simulation
import gymnasium as gym

# Load the classic Mountain Car environment
env = gym.make("MountainCar-v0")

ModuleNotFoundError: No module named 'gymnasium'

In [ ]:
# Define how many "bins" or "buckets" we want for position and velocity
num_bins = 20
# Create evenly spaced boundaries for the position range [-1.2, 0.6]
pos_bins = np.linspace(-1.2, 0.6, num_bins)
# Create evenly spaced boundaries for the velocity range [-0.07, 0.07]
vel_bins = np.linspace(-0.07, 0.07, num_bins)

In [ ]:
# Step 1: Create two separate Q-tables initialized to zero: [pos_bucket, vel_bucket, action]
Q1 = np.zeros((num_bins, num_bins, 3))
Q2 = np.zeros((num_bins, num_bins, 3))

# Hyperparameters
alpha = 0.1     # Learning rate (how fast we update our cheat sheet)
gamma = 0.99    # Discount factor (future value focus)
epsilon = 0.1   # Exploration rate (10% chance of random moves)

In [ ]:
# Helper function to convert raw decimal state to integer bucket indices
def get_discretized_state(state):
    # Retrieve raw coordinates
    pos, vel = state
    # Find which bin index the values fall into
    pos_idx = np.digitize(pos, pos_bins) - 1
    vel_idx = np.digitize(vel, vel_bins) - 1
    # Ensure indices stay within our array boundaries [0, num_bins-1]
    return int(np.clip(pos_idx, 0, num_bins-1)), int(np.clip(vel_idx, 0, num_bins-1))

In [ ]:
# Train the agent over 1000 short episodes
for episode in range(1000):
    # Reset the environment and get the starting continuous state
    raw_state, info = env.reset()
    state = get_discretized_state(raw_state)
    done = False

    while not done:
        # Step 2: Choose action using epsilon-greedy strategy
        if np.random.rand() < epsilon:
            action = env.action_space.sample()  # Explore: random action
        else:
            # Exploit: choose best action based on the sum of both tables
            action = np.argmax(Q1[state[0], state[1]] + Q2[state[0], state[1]])

        # Execute the chosen action in the simulator
        next_raw_state, reward, terminated, truncated, _ = env.step(action)
        next_state = get_discretized_state(next_raw_state)
        done = terminated or truncated

        # Step 3: Randomly update Q1 or Q2 to eliminate maximization bias
        if np.random.rand() < 0.5:
            # Find the best action according to Q1
            best_next_action = np.argmax(Q1[next_state[0], next_state[1]])
            # Use Q2 to evaluate that action to get a realistic score
            target = reward + gamma * Q2[next_state[0], next_state[1], best_next_action]
            # Update Q1 entry
            Q1[state[0], state[1], action] += alpha * (target - Q1[state[0], state[1], action])
        else:
            # Find the best action according to Q2
            best_next_action = np.argmax(Q2[next_state[0], next_state[1]])
            # Use Q1 to evaluate that action to get a realistic score
            target = reward + gamma * Q1[next_state[0], next_state[1], best_next_action]
            # Update Q2 entry
            Q2[state[0], state[1], action] += alpha * (target - Q2[state[0], state[1], action])

        # Move to the next state
        state = next_state

print("Double Q-Tables trained successfully!")

In [ ]:
print("Sample Q-values for starting position [-0.5, 0.0]:")
start_idx = get_discretized_state([-0.5, 0.0])
print("Table 1:", Q1[start_idx[0], start_idx[1]])
print("Table 2:", Q2[start_idx[0], start_idx[1]])